### DLT PIPELINE: SILVER LAYER (02_silver_transformation)

In [0]:
# -------------------------------------------------------------------------
# Purpose: Reads from BRONZE -> Writes to SILVER
# UPGRADE: Handles NULLS in non-critical columns (Brand, Category)
# -------------------------------------------------------------------------

import dlt
from pyspark.sql.functions import col, to_date, lit, when, current_timestamp, coalesce

BRONZE_TABLE = "ecommerce_analytics_dev.bronze_layer.events_raw"

rules = {
    "valid_user_id": "user_id IS NOT NULL",
    "valid_event_time": "event_time IS NOT NULL",
    "valid_price": "price >= 0"
}

# Silver table: cleans and validates raw events from bronze layer.
# - Drops rows with missing user_id, event_time, or negative price (critical errors).
# - Fills NULLs in non-critical columns (brand, category, session) with default values.
# - Adds audit columns and deduplicates events.
@dlt.table(
    name="events_silver",
    comment="Cleans and validates raw events: drops critical errors, fills missing non-critical attributes, deduplicates, and tags data quality.",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_user_id", rules["valid_user_id"])
@dlt.expect_or_drop("valid_event_time", rules["valid_event_time"])
@dlt.expect_or_drop("valid_price", rules["valid_price"]) 
def events_silver():
    return (
        spark.readStream
        .option("maxBytesPerTrigger", "1g") 
        .table(BRONZE_TABLE) 
        .select(
            # 1. Critical Columns (Must be valid or dropped by Expectation)
            col("event_time").cast("timestamp"),
            col("user_id").cast("integer"),
            col("price").cast("double"),
            col("product_id").cast("integer"),
            col("event_type").cast("string"),
            
            # 2. Non-Critical Columns (Fix NULLs here!)
            # If brand is null, make it 'Unknown'. If category is null, make it 'n/a'
            coalesce(col("category_code"), lit("n/a")).alias("category_code"),
            coalesce(col("brand"), lit("Unknown")).alias("brand"),
            coalesce(col("user_session"), lit("unknown_session")).alias("user_session"),
            
            # Handle numeric nulls safely (optional, usually -1 for ID)
            coalesce(col("category_id"), lit(-1)).cast("long").alias("category_id"),

            # 3. Audit Columns
            col("source_file"),
            col("ingestion_timestamp"),
            lit("silver").alias("data_quality_tag")
        )
        .dropDuplicates(["event_time", "user_id", "product_id", "event_type"])
        .withColumn("event_date", to_date(col("event_time")))
    )

# Quarantine table: captures records that fail critical data quality checks.
# - Stores failed records with reason for violation for further inspection.
@dlt.table(
    name="events_quarantine",
    comment="Captures records failing critical data quality checks for inspection.",
    table_properties={"quality": "quarantine"}
)
def events_quarantine():
    return (
        spark.readStream
        .option("maxBytesPerTrigger", "1g")
        .table(BRONZE_TABLE)
        .select(
            col("event_time").cast("timestamp"),
            col("user_id").cast("integer"),
            col("price").cast("double"),
            col("event_type"),
            col("_rescued_data"), 
            col("source_file"),
            current_timestamp().alias("quarantine_timestamp")
        )
        .filter(
            f"NOT ({rules['valid_user_id']}) OR "
            f"NOT ({rules['valid_event_time']}) OR "
            f"NOT ({rules['valid_price']})"
        )
        .withColumn("violation_reason", 
            when(col("user_id").isNull(), "Missing User ID")
            .when(col("event_time").isNull(), "Missing Timestamp")
            .when(col("price") < 0, "Negative Price")
            .otherwise("Unknown")
        )
    )